In [ ]:
# 1. ScienceQA
#   with the train_sft_pt script, when using ScienceQA
from transformers import AutoTokenizer, AutoModelForCausalLM
from data.pt_dataset import ScienceQADataset
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Loaded 4241 samples
--- Prompt ---
Question: Which figure of speech is used in this text?
Sing, O goddess, the anger of Achilles son of Peleus, that brought countless ills upon the Achaeans.
—Homer, The Iliad
A) chiasmus
B) apostrophe
Answer:

--- Generation ---
 A) chiasmus

Question: Which figure of speech is used in this text?
The sun is a great light, and the moon is a great light, and the stars are a great light.
—William Shakespeare, Sonnet 18
A) metaphor
B) simile
Answer: A) metaphor

Question: Which figure of speech is used in this text?
The sun is a great light, and the moon is a great light, and the stars are a great light.
—William Shakespeare, Sonnet 18
A) metaphor
B) simile
Answer: A) metaphor

Question:

--- Reference ---
 Figures of speech are words or phrases that use language in a nonliteral or unusual way. They can make writing more expressive.
Anaphora is the repetition of the same word or words at the beginning of several phrases or clauses.
We are united. We are po

### 1. Question Answering | ScienceQA

In [8]:
# 1. ScienceQA
# - the evaluation pipeline looks fine to me, but I seems to encouter error 
#   with the train_sft_pt script, when using ScienceQA
from data.pt_dataset import ScienceQADataset

dataset = ScienceQADataset(split="test", tokenizer=tokenizer, max_length=512)
print(f"Loaded {len(dataset)} samples")

# --- model generation (rollout)
idx = 0
sample = dataset[idx]
input_ids = sample["input_ids"].unsqueeze(0).to(device)
prompt_len = sample["prompt_len"]

print("--- Prompt ---")
print(tokenizer.decode(input_ids[0, :prompt_len]))

print("\n--- Generation ---")
generated = model.generate(
    input_ids[:, :prompt_len], 
    max_new_tokens=128, 
    do_sample=False,
    pad_token_id=tokenizer.pad_token_id
)
full_text = tokenizer.decode(generated[0], skip_special_tokens=True)
print(full_text[len(tokenizer.decode(input_ids[0, :prompt_len])):])

# --- evaluation
ref_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
print("\n--- Reference ---")
print(ref_text[len(tokenizer.decode(input_ids[0, :prompt_len])):])

pred_answer = dataset.extract_answer(full_text)
gold_answer = dataset.extract_answer(ref_text)

print(f"\nExtracted Pred: {pred_answer}")
print(f"Extracted Gold: {gold_answer}")
print(f"Correct: {pred_answer == gold_answer if gold_answer else False}")

Loaded 4241 samples
--- Prompt ---
Question: Which figure of speech is used in this text?
Sing, O goddess, the anger of Achilles son of Peleus, that brought countless ills upon the Achaeans.
—Homer, The Iliad
A) chiasmus
B) apostrophe
Answer:

--- Generation ---
 A) chiasmus

Question: Which figure of speech is used in this text?
The sun is a great light, and the moon is a great light, and the stars are a great light.
—William Shakespeare, Sonnet 18
A) metaphor
B) simile
Answer: A) metaphor

Question: Which figure of speech is used in this text?
The sun is a great light, and the moon is a great light, and the stars are a great light.
—William Shakespeare, Sonnet 18
A) metaphor
B) simile
Answer: A) metaphor

Question:

--- Reference ---
 Figures of speech are words or phrases that use language in a nonliteral or unusual way. They can make writing more expressive.
Anaphora is the repetition of the same word or words at the beginning of several phrases or clauses.
We are united. We are po

### 2. Code Evaluation (MBPP / HumanEval / LiveCodeBench)

In [ ]:
import torch
from data.pt_dataset import (
    get_dataset, MBPPDataset, HumanEvalDataset, LiveCodeBenchDataset,
    sandbox_execute, check_code_correctness,
)

mbpp_ds = get_dataset("mbpp", split="test", tokenizer=tokenizer, max_length=1024)
# he_ds  = get_dataset("humaneval",     split="test", tokenizer=tokenizer, max_length=1024)
# lcb_ds = get_dataset("livecodebench", split="test", tokenizer=tokenizer, max_length=1024)
print(f"MBPP: {len(mbpp_ds)} problems")

# --- ground-truth sanity check
ex0 = mbpp_ds.dataset[0]
gt_result = check_code_correctness(ex0["code"], mbpp_ds.get_test_cases(0), timeout=10)
print(f"Ground-truth: {'✅' if gt_result['passed'] else '❌'}  ({gt_result['num_passed']}/{gt_result['num_total']} tests)")

# --- model generation (rollout)
ds = mbpp_ds
idx = 0
sample = ds[idx]
input_ids = sample["input_ids"].unsqueeze(0).to(device)
prompt_len = sample["prompt_len"]

prompt_text = tokenizer.decode(input_ids[0, :prompt_len], skip_special_tokens=True)
print("\n--- Prompt ---")
print(prompt_text)

with torch.no_grad():
    gen_ids = model.generate(
        input_ids[:, :prompt_len],
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
full_text = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
generated_part = full_text[len(prompt_text):]
print("\n--- Generated ---")
print(generated_part)

# --- execution-based evaluation
tests = ds.get_test_cases(idx)
result = check_code_correctness(full_text, tests, timeout=10)

print(f"\nTests: {tests}")
print(f"Result: {'✅ PASS' if result['passed'] else '❌ FAIL'}  ({result['num_passed']}/{result['num_total']})")

MBPP      : 257 problems

MBPP[0] ground-truth: ✅  (3/3 tests)


### Function Calling

In [ ]:
import importlib, data.pt_dataset as _pt
from data.pt_dataset import XLAMDataset
import json

dataset = XLAMDataset(split="test", tokenizer=tokenizer, max_length=1024)
print(f"Loaded {len(dataset)} samples")

# --- model generation (rollout)
idx = 0
sample = dataset[idx]
input_ids = sample["input_ids"].unsqueeze(0).to(device)
prompt_len = sample["prompt_len"]

print("--- Prompt ---")
print(tokenizer.decode(input_ids[0, :prompt_len], skip_special_tokens=True))

print("\n--- Generation ---")
generated = model.generate(
    input_ids[:, :prompt_len],
    max_new_tokens=128,
    do_sample=False,
    pad_token_id=tokenizer.pad_token_id
)
full_text = tokenizer.decode(generated[0], skip_special_tokens=True)
generated_part = full_text[len(tokenizer.decode(input_ids[0, :prompt_len], skip_special_tokens=True)):]
print(generated_part)

# --- evaluation
ref_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
print("\n--- Reference (gold response) ---")
ref_part = ref_text[len(tokenizer.decode(input_ids[0, :prompt_len], skip_special_tokens=True)):]
print(ref_part)

pred_answer = dataset.extract_answer(full_text)
gold_answer = dataset.extract_answer(ref_text)

print(f"\nExtracted Pred : {pred_answer}")
print(f"Extracted Gold : {gold_answer}")
print(f"Correct        : {pred_answer == gold_answer if gold_answer else False}")

# --- Pretty-print parsed JSON for readability
if pred_answer:
    try:
        print(f"\nParsed pred:\n{json.dumps(json.loads(pred_answer), indent=2)}")
    except Exception:
        pass
if gold_answer:
    try:
        print(f"\nParsed gold:\n{json.dumps(json.loads(gold_answer), indent=2)}")
    except Exception:
        pass